
<link rel="stylesheet" href="berkeley.css">

<h1 class="cal cal-h1">Lecture 03: PCA and Linear Algebra Review – CS 189, Fall 2026</h1>

**Demonstration: making sense of congressional votes.**

We use PCA today before we understand it. By the end of this notebook we will have taken a
441 x 41 table of votes, reduced it to 441 x 2, and recovered a property of Congress that was
never supplied to the algorithm. The remainder of the lecture explains why this works.

In [ ]:
import numpy as np
import pandas as pd
import yaml
from datetime import datetime
import plotly.express as px
# # Uncomment for HTML Export
# import plotly.io as pio
# pio.renderers.default = "notebook_connected"

## Congressional Vote Records

Let's examine how the House of Representatives (of the 116th Congress, 1st session) voted in the month of **September 2019**.

From the [U.S. Senate website](https://www.senate.gov/reference/Index/Votes.htm):

> Roll call votes occur when a representative or senator votes "yea" or "nay," so that the names of members voting on each side are recorded. A voice vote is a vote in which those in favor or against a measure say "yea" or "nay," respectively, without the names or tallies of members voting on each side being recorded.

The data, compiled from ProPublica [source](https://github.com/eyeseast/propublica-congress), is a "skinny" table of data where each record is a single vote by a member across any roll call in the 116th Congress, 1st session, as downloaded in February 2020. The member of the House, whom we'll call **legislator**, is denoted by their bioguide alphanumeric ID in http://bioguide.congress.gov/.

In [ ]:
votes = pd.read_csv('data/votes.csv')
votes = votes.astype({"roll call": str})
votes

In [ ]:
votes['vote'].value_counts()

This is a "skinny" table, with one row per (member, roll call). To treat each legislator as a
**datapoint**, we pivot so that each row is a legislator and each column is a roll call.
We record a `1` for a Yes vote and a `0` otherwise.

In [ ]:
def was_yes(s):
    return 1 if s.iloc[0] == "Yes" else 0

vote_pivot = votes.pivot_table(index='member',
                               columns='roll call',
                               values='vote',
                               aggfunc=was_yes,
                               fill_value=0)
print(vote_pivot.shape)
vote_pivot.head()

So our data matrix $X$ has **441 rows** (legislators) and **41 columns** (roll calls).

Each legislator is a point in 41-dimensional space.

In [ ]:
X = vote_pivot.to_numpy()
X.shape

## A first attempt: plotting the raw columns

We can only look at two dimensions at a time on a screen. The obvious approach is to select two
columns and produce a scatter plot.

In [ ]:
px.scatter(vote_pivot, x='555', y='553',
           title='Two roll calls at a time', width=700, height=500)

That is four dots.

Every legislator sits at one of four corners, so 441 points collapse onto 4 visible marks. Even
if we jittered them apart, there would be $\binom{41}{2} = 820$ such plots to inspect.

**Selecting two of the original columns is not adequate.** We require two *new* columns,
constructed from all 41.

## PCA in three lines

In [ ]:
from sklearn.decomposition import PCA

model = PCA(n_components=2)
Z = model.fit_transform(vote_pivot)
Z.shape

Each of the 441 legislators is now described by **2 numbers rather than 41**. We plot them
below.

In [ ]:
px.scatter(x=Z[:, 0], y=Z[:, 1],
           labels={'x': 'z1', 'y': 'z2'},
           title='Vote data projected onto 2 dimensions',
           width=800, height=600, opacity=0.7)

Two clusters appear, although nothing in the input indicated that clusters should exist. The
algorithm never saw a party label, only zeros and ones.

We now bring in the identity of each legislator, from
[unitedstates/congress-legislators](https://github.com/unitedstates/congress-legislators).

In [ ]:
# Static copy of the 2019 roster, so it matches our voting data.
legislators_data = yaml.safe_load(open('data/legislators-2019.yaml'))

def to_date(s):
    return datetime.strptime(s, '%Y-%m-%d')

legs = pd.DataFrame(
    columns=['leg_id', 'first', 'last', 'state', 'chamber', 'party', 'birthday'],
    data=[[x['id']['bioguide'],
           x['name']['first'],
           x['name']['last'],
           x['terms'][-1]['state'],
           x['terms'][-1]['type'],
           x['terms'][-1]['party'],
           to_date(x['bio']['birthday'])] for x in legislators_data])
legs['age'] = 2024 - legs['birthday'].dt.year
legs.head()

In [ ]:
vote_2d = pd.DataFrame(Z, index=vote_pivot.index, columns=['z1', 'z2'])
vote_2d = vote_2d.join(legs.set_index('leg_id'))
vote_2d.head()

In [ ]:
px.scatter(vote_2d, x='z1', y='z2', color='party',
           title='Vote data, colored by party (PCA never saw this column)',
           width=800, height=600, opacity=0.7,
           color_discrete_map={'Democrat': 'blue', 'Republican': 'red', 'Independent': 'green'},
           hover_data=['first', 'last', 'state'],
           render_mode='svg')

The structure that PCA found corresponds to party affiliation.

There is substantial overplotting, since many legislators vote identically and therefore land on
exactly the same point. We add jitter to reveal the density.

In [ ]:
rng = np.random.default_rng(42)
vote_2d['z1_jittered'] = vote_2d['z1'] + rng.normal(0, 0.1, len(vote_2d))
vote_2d['z2_jittered'] = vote_2d['z2'] + rng.normal(0, 0.1, len(vote_2d))

px.scatter(vote_2d, x='z1_jittered', y='z2_jittered', color='party', size='age',
           title='Vote data (jittered)',
           width=800, height=600, opacity=0.7, size_max=10,
           color_discrete_map={'Democrat': 'blue', 'Republican': 'red', 'Independent': 'green'},
           hover_data=['first', 'last', 'state', 'party'])

How far does this go? If we use only the **sign of the first coordinate**, how often does it
agree with party affiliation?

In [ ]:
labeled = vote_2d.dropna(subset=['party'])
guess = np.where(labeled['z1'] > 0, 'Democrat', 'Republican')
(guess == labeled['party']).mean()

A single number per legislator, derived from all 41, recovers party affiliation for
approximately 98% of the House.

## How much information was discarded?

We replaced 41 columns with 2. The attribute `explained_variance_ratio_` reports the fraction of
the spread in the data accounted for by each new coordinate.

In [ ]:
model.explained_variance_ratio_

In [ ]:
model.explained_variance_ratio_.sum()

The first coordinate alone accounts for roughly 80%. The full profile is shown below.

In [ ]:
model10 = PCA(n_components=10).fit(vote_pivot)

px.line(y=model10.explained_variance_ratio_, markers=True,
        labels={'x': 'component', 'y': 'fraction of total spread'},
        title='Spread accounted for by each component',
        width=700, height=450)

There is a sharp decrease after the first component, followed by a long flat tail. This shape is
what makes the two-dimensional plot trustworthy: no third direction accounts for an appreciable
share of the spread.

Note that the data matrix is *not* actually low rank:

In [ ]:
np.linalg.matrix_rank(X - X.mean(axis=0))

The rank is 41, which is full. This is therefore not a case of exactly redundant columns that
may be deleted. The data is only **approximately** low dimensional, and that distinction is
central to what follows.

## What the model consists of

The call to `fit` estimated something. We inspect it below.

In [ ]:
model.components_.shape

In [ ]:
model.components_

**Two rows of 41 numbers.** This is the entire model.

Each row is a set of weights over the 41 roll calls, and a legislator's new coordinates are the
dot products of their voting record with these two rows:

In [ ]:
w1 = model.components_[0]
Xc = X - X.mean(axis=0)          # PCA centers the data internally

# sklearn's answer for the first legislator, vs. a dot product we compute ourselves
print(Z[0, 0], Xc[0] @ w1)

In [ ]:
px.bar(x=vote_pivot.columns, y=w1,
       labels={'x': 'roll call', 'y': 'weight in the first component'},
       title='The first row of components_',
       width=900, height=400)

In [ ]:
# Center each column: subtract the House-wide Yes rate for that roll call.
vote_pivot_centered = vote_pivot - vote_pivot.mean()

# Per party, the average deviation from the House-wide Yes rate on each roll call.
party_yes_deviation = vote_pivot_centered.join(labeled['party']).groupby('party').mean()

party_yes_deviation_long = (party_yes_deviation
                            .reset_index()
                            .melt(id_vars='party',
                                  var_name='roll call',
                                  value_name='Yes Rate Centered'))

fig = px.bar(party_yes_deviation_long,
             x='roll call', y='Yes Rate Centered',
             facet_row='party', color='party',
             color_discrete_map={'Democrat': 'blue', 'Republican': 'red', 'Independent': 'green'},
             title='Party Yes rate relative to the House average, by roll call',
             width=900, height=800)
fig.for_each_annotation(lambda a: a.update(text=a.text.split('=')[-1]))  # 'party=Democrat' -> 'Democrat'
fig.update_layout(showlegend=False)  # the facet titles already name the party
fig

The method therefore reduces to a single question: **how do we find the right $k$ rows of
length $d$?**

- How are those rows determined?
- In what sense are they the *optimal* choice?
- Why does projecting onto them preserve the structure of interest?

These are the subject of the remainder of the lecture.

## Compressing images: PCA on Fashion-MNIST

The congressional votes were 41-dimensional. We now apply the same three lines to a
dataset where each datapoint has 784 dimensions and, unlike a voting record, can be looked
at directly. [Fashion-MNIST](https://github.com/zalandoresearch/fashion-mnist) is 60,000
grayscale 28 x 28 photographs of clothing in 10 categories.

In [ ]:
# Fetch the Data
import torchvision
data = torchvision.datasets.FashionMNIST(root='data', train=True, download=True)

# Preprocess the data into numpy arrays
images = data.data.numpy().astype(float)
targets = data.targets.numpy() # integer encoding of class labels
class_dict = {i:class_name for i,class_name in enumerate(data.classes)}
labels = np.array([class_dict[t] for t in targets]) # raw class labels
n = len(images)

print("Loaded FashionMNIST dataset with {} samples.".format(n))
print("Classes: {}".format(class_dict))
print("Image shape: {}".format(images[0].shape))
print("Image dtype: {}".format(images[0].dtype))
print("Image 0:\n", images[0])

In [ ]:
px.imshow(images[0], color_continuous_scale='gray_r') 

In [ ]:
def show_images(images, max_images=40, ncols=5, labels = None):
    """Visualize a subset of images from the dataset.
    Args:
        images (np.ndarray): Array of images to visualize [img,row,col].
        max_images (int): Maximum number of images to display.
        ncols (int): Number of columns in the grid.
        labels (np.ndarray, optional): Labels for the images, used for facet titles.
    Returns:
        plotly.graph_objects.Figure: A Plotly figure object containing the images.
    """
    n = min(images.shape[0], max_images) # number of images to show
    px_height = 220 # height of each image in pixels
    fig = px.imshow(images[:n, :, :], color_continuous_scale='gray_r', 
                    facet_col = 0, facet_col_wrap=ncols,
                    height = px_height * int(np.ceil(n/ncols)))
    fig.update_layout(coloraxis_showscale=False)
    if labels is not None:
        # Extract the facet number and replace with the label.
        fig.for_each_annotation(lambda a: a.update(text=labels[int(a.text.split("=")[-1])]))
    return fig

In [ ]:
show_images(images, 20, labels=labels)

### Each image is a datapoint in 784-dimensional space

The voting data gave us one row per legislator and one column per roll call. We do the same
thing here: one row per image, one column per **pixel**. A 28 x 28 image becomes a single row of
$28 \times 28 = 784$ numbers.

In [ ]:
X_img = images.reshape(n, -1)   # (60000, 28, 28) -> (60000, 784)
X_img.shape

PCA centers the data before it does anything else, so the first thing it computes is the mean of
those 60,000 rows. Reshaped back to 28 x 28, the mean is itself an image.

In [ ]:
mean_image = X_img.mean(axis=0)

px.imshow(mean_image.reshape(28, 28), color_continuous_scale='gray_r',
          title='The average of all 60,000 images', width=400, height=400)

### The principal components are also images

For the votes, `components_` was a $k \times 41$ matrix, and each row was a set of weights over
the 41 roll calls. Here it is a $k \times 784$ matrix, and each row is a set of weights over the
784 pixels. **A row of 784 numbers can be reshaped into a 28 x 28 picture**, so we can look
directly at the model.

We fit 200 components once, and use the leading $k$ of them below. Because the components are
nested, `components_[:k]` is exactly what `PCA(n_components=k)` would have produced.

In [ ]:
pca_img = PCA(n_components=200).fit(X_img)
pca_img.components_.shape

In [ ]:
n_show = 10
comps = pca_img.components_[:n_show].reshape(n_show, 28, 28)

fig = px.imshow(comps, facet_col=0, facet_col_wrap=5,
                color_continuous_scale='RdBu_r', color_continuous_midpoint=0,
                height=440, title='The first 10 principal components, viewed as images')
fig.for_each_annotation(lambda a: a.update(text=f"PC {int(a.text.split('=')[-1]) + 1}"))
fig.update_layout(coloraxis_showscale=False)
fig

Red is a positive weight and blue is a negative one. These are not garments; they are
**contrasts**. PC 1 separates wide dark regions from narrow ones, which is roughly the
distinction between a shirt and a shoe. Later components encode sleeves, straps, and the gap
between trouser legs. The overall sign of each component is arbitrary.

### How many components do we need?

In [ ]:
cumvar = np.cumsum(pca_img.explained_variance_ratio_)

px.line(x=np.arange(1, 201), y=cumvar,
        labels={'x': 'number of components k', 'y': 'cumulative fraction of spread'},
        title='Spread captured by the first k components', width=750, height=450)

The curve rises steeply and then flattens: 50 of the 784 directions account for 86% of the
spread, and 200 account for 95%. Compare this to the votes, where a *single* component captured
80%. Images are low dimensional, but not nearly as aggressively so.

### Reconstruction

Compression is only useful if we can get the image back. Keeping $k$ scores and then
undoing the projection gives

$$\hat{x} = \bar{x} + \sum_{j=1}^{k} z_j w_j$$

a picture rebuilt as the mean image plus a weighted sum of $k$ component images.

In [ ]:
Z_img = pca_img.transform(X_img)   # (60000, 200) scores

def reconstruct(k, rows):
    """Rebuild images from only their first k scores."""
    return Z_img[rows, :k] @ pca_img.components_[:k] + mean_image

In [ ]:
ks = [1, 2, 5, 10, 25, 50, 100, 200]
i = 0   # the ankle boot from the top of this section

ladder = np.vstack([X_img[i]] + [reconstruct(k, [i]) for k in ks])

# Reconstructions can fall slightly outside [0, 255], so we clip them for display.
show_images(np.clip(ladder, 0, 255).reshape(-1, 28, 28), ncols=3,
            labels=['original (784)'] + [f'k = {k}' for k in ks])

One number produces a dark blob. Ten produce something identifiable as a boot. By 50 the
silhouette and the shading are right, and the remaining 734 dimensions mostly carry
texture and noise.

Below we do the same at $k = 50$ for eight random images. The top row is the original,
the bottom row is 50 numbers.

In [ ]:
rng_img = np.random.default_rng(189)
rows = rng_img.choice(n, 8, replace=False)
k = 50

side_by_side = np.vstack([X_img[rows], reconstruct(k, rows)])
show_images(np.clip(side_by_side, 0, 255).reshape(-1, 28, 28), ncols=8,
            labels=[labels[r] for r in rows] + [f'k = {k}' for _ in rows])

### What did that actually save?

To store the whole dataset we need the $n \times k$ table of scores, plus the basis we need in
order to decode it: the $k \times 784$ components and the 784-pixel mean. The basis is paid for
**once**, no matter how many images we compress.

In [ ]:
d = X_img.shape[1]

for k in [10, 50, 100]:
    stored = n * k + k * d + d
    rmse = np.sqrt(((reconstruct(k, slice(None)) - X_img) ** 2).mean())
    print(f"k = {k:3d} | scores {n*k:>9,} + basis {k*d + d:>7,} = {stored:>9,} numbers "
          f"| {n*d/stored:5.1f}x smaller | RMSE {rmse:5.1f}")

At $k = 50$ the dataset is **15x smaller** and the images survive. The basis is only 1.3% of the
stored bytes, so essentially all of the cost is the 50 numbers per image.

This is lossy compression of the same general kind as JPEG. The difference is that JPEG uses a
fixed, universal basis (cosines), while PCA **learns** a basis from this particular collection of
images. That is why the components above look like clothing contrasts rather than generic
ripples, and it is also why the basis only compresses images that resemble the training set.

## Can we run the decoder backwards to invent new clothes?

The reconstruction step $\hat{x} = \bar{x} + \sum_j z_j w_j$ turns 50 numbers into an image, and
it does not care where those numbers came from. So here is a tempting idea: instead of taking
$z$ from a real image, **make $z$ up**, and see what comes out.

For this to work, the made-up $z$ has to look like the $z$ of a real image. So first we
look at how the real scores are distributed.

In [ ]:
sub = np.random.default_rng(189).choice(n, 4000, replace=False)
scores_2d = pd.DataFrame({'z1': Z_img[sub, 0], 'z2': Z_img[sub, 1], 'class': labels[sub]})

px.scatter(scores_2d, x='z1', y='z2', color='class', opacity=0.6,
           title='The first two scores of 4,000 images', width=850, height=600)

This is not one cloud. Footwear sits on the left, tops sit on the upper right, trousers hang
below, bags sit on top. The score distribution is **multimodal**, and there are wide empty
regions between the groups.

Let us ignore that for a moment and do the simplest thing: fit a single Gaussian to the 50
scores, draw from it, and decode.

In [ ]:
k = 50
Zk = Z_img[:, :k]

def sample_images(Z_ref, n_samples, rng):
    """Fit one Gaussian to the scores in Z_ref, draw from it, and decode into images."""
    draws = rng.multivariate_normal(Z_ref.mean(axis=0), np.cov(Z_ref.T), n_samples)
    return draws @ pca_img.components_[:Z_ref.shape[1]] + mean_image

fake = sample_images(Zk, 16, np.random.default_rng(0))
show_images(np.clip(fake, 0, 255).reshape(-1, 28, 28), ncols=8, max_images=16)

These are garment-shaped smudges. Several are two items at once: a sleeve fading into a trouser
leg, a shoe ghosted over a shirt. They have the *statistics* of the dataset without being
plausible members of it.

The scatter plot above explains why. A single Gaussian is one blob, so most of its mass lands in
the empty space *between* the clusters, and a point halfway between "sneaker" and "pullover"
decodes to a superposition of the two. We can check that the samples really are landing in
unoccupied territory by measuring how far each one is from the nearest real image.

In [ ]:
rng_nn = np.random.default_rng(1)
reference = Zk[rng_nn.choice(n, 8000, replace=False)]
real_pts  = Zk[rng_nn.choice(n, 200, replace=False)]
fake_pts  = rng_nn.multivariate_normal(Zk.mean(axis=0), np.cov(Zk.T), 200)

def nn_distance(query, reference):
    """Distance from each query point to its closest neighbour in reference."""
    sq = ((query[:, None, :] - reference[None, :, :]) ** 2).sum(axis=2)
    return np.sqrt(sq.min(axis=1))

print(f"real image  -> nearest real image: {np.median(nn_distance(real_pts, reference)):.0f}")
print(f"fake sample -> nearest real image: {np.median(nn_distance(fake_pts, reference)):.0f}")

A real image has a real neighbour roughly twice as close. The samples are not near the data;
they are in the gaps.

The fix follows directly from the diagnosis. The problem was fitting **one** blob to **ten**
clusters, so we fit one Gaussian per class instead, still in the same 50-dimensional score
space, and still decoding with the same components.

In [ ]:
rng_cls = np.random.default_rng(0)
chosen = [0, 7, 8, 1]   # T-shirt/top, Sneaker, Bag, Trouser

per_class = np.vstack([sample_images(Zk[targets == c], 4, rng_cls) for c in chosen])

show_images(np.clip(per_class, 0, 255).reshape(-1, 28, 28), ncols=4, max_images=16,
            labels=[class_dict[c] for c in chosen for _ in range(4)])

**These work.** Each row is four garments that do not exist in the dataset, and they are
recognizably sneakers, t-shirts, bags, and trousers, with varied heights, widths, and shading.
They are blurry, because 50 components cannot represent a sharp edge and because a Gaussian is
still only an approximation of a class, but they are plausible items rather than superpositions.

So the honest answer is: **the decoder is fine, and the hard part is knowing which $z$ to
feed it.** Sampling works exactly as well as our model of the score distribution does.

### One caution about what the subspace can do

It is tempting to read the score space as a space of *concepts*, where moving from one image to
another should morph a shirt into a shoe. It cannot, and the reason is that the map from $z$ to
pixels is **linear**. Interpolating between two images in score space is algebraically identical
to cross-fading the two images in pixel space.

In [ ]:
a = np.where(targets == 0)[0][0]   # a t-shirt
b = np.where(targets == 7)[0][0]   # a sneaker

t = np.linspace(0, 1, 8)[:, None]
path = (1 - t) * Zk[a] + t * Zk[b]
blend = path @ pca_img.components_[:k] + mean_image

show_images(np.clip(blend, 0, 255).reshape(-1, 28, 28), ncols=8, max_images=8)

In [ ]:
# The same path, computed instead by blending pixels and then projecting. Identical.
pixel_blend = (1 - t) * X_img[a] + t * X_img[b]
projected = (pixel_blend - mean_image) @ pca_img.components_[:k].T @ pca_img.components_[:k] + mean_image

np.abs(blend - projected).max()

The shirt does not become a shoe; it dissolves while a shoe appears underneath. Linearity is
what makes PCA cheap to fit, easy to interpret, and provably optimal in the sense we are about to
define, and it is also precisely what stops it from being a generative model of images. Getting
a genuine morph requires a decoder that is *not* restricted to a linear subspace.

### Summary of this section

- Each image is a point in $\mathbb{R}^{784}$; PCA finds a $k$-dimensional subspace it nearly lies in.
- The components are pictures, and reconstruction is the mean image plus a weighted sum of them.
- $k = 50$ compresses the dataset 15x with the content of the images intact.
- Decoding invented scores does generate new clothing, but only once the score distribution is
  modeled per class. A single Gaussian samples the empty space between clusters.
- The subspace is linear, so interpolation is a cross-fade, not a morph.